In [ ]:
import pandas as pd
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
import importlib



In [25]:
BASE_DIR = Path.cwd().parent

STAGE2_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "stage2"
    / "stage2_df.csv"
)

print("Current directory:", Path.cwd())
print("Stage 2 file:", STAGE2_PATH.resolve())
print("File exists:", STAGE2_PATH.exists())

Current directory: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\notebooks
Stage 2 file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_df.csv
File exists: True


In [26]:
if not STAGE2_PATH.exists():
    raise FileNotFoundError(
        f"Stage 2 file not found: {STAGE2_PATH.resolve()}"
    )

stage2_df = pd.read_csv(STAGE2_PATH)

print("Shape:", stage2_df.shape)
print("Columns:", stage2_df.columns.tolist())

display(stage2_df.head())

Shape: (5258, 40)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known']


,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,title_normalized,title_punct_normalized,title_group,model_key,match_rule,confidence,family,storage_known,ram_known,condition_known
0,Honor Magic V5,80000.0,اوبن اير مول، مدينتي,/ad/honor-magic-v5-ID209010621.html,209010621,legacy,honor,NaN,NaN,NaN,...,honor magic v5,honor magic v5,honor magic v5,honor magic v5,honor:rule_04,NaN,magic,False,False,False
1,شاومي 17 الترا xiaomi 17 ultra,70000.0,التجمع الخامس، القاهرة الجديدة,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-17-%D8%A7%D...,208281312,legacy,xiaomi,NaN,NaN,NaN,...,شاومي17الترا xiaomi17ultra,شاومي17الترا xiaomi17ultra,شاومي17الترا xiaomi17ultra,xiaomi 17 ultra,xiaomi:rule_17,NaN,xiaomi,False,False,False
2,Apple - iPhone 17 Pro,75000.0,محور 26 يوليو، 6 اكتوبر,/ad/apple-iphone-17-pro-ID208973056.html,208973056,legacy,apple,NaN,NaN,NaN,...,apple iphone17pro,apple iphone17pro,apple iphone17pro,apple iphone 17 pro,apple:model_pattern_06,1.0,iphone,False,False,False
3,Honor 500 pro,35000.0,6 اكتوبر، الجيزة,/ad/honor-500-pro-ID208029599.html,208029599,legacy,honor,NaN,NaN,NaN,...,honor500pro,honor500pro,honor500pro,honor 500 pro,honor:rule_07,NaN,honor numeric,False,False,False
4,آبل - آيفون 15 برو ماكس,58000.0,ستيلا هليوبوليس، مدينة الشروق,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,208989761,legacy,apple,NaN,NaN,NaN,...,ابل ايفون15برو ماكس,ابل ايفون15برو ماكس,ابل ايفون15برو ماكس,apple iphone 15 pro max,apple:model_pattern_02,1.0,iphone,False,False,False


# Baseline

A baseline is a simple method used to predict prices. Its purpose is to answer:

How well can we predict phone prices using a simple approach, before introducing more complex models?

We need a starting point to determine whether a new model actually improves prediction accuracy.

to understand which information contributes most to price prediction, and make sure that future improvements are supported by evidence.

In [27]:
baseline_df = stage2_df.loc[stage2_df["price"].notna() & stage2_df["title_group"].notna()].copy()
baseline_df["price"] = pd.to_numeric(baseline_df["price"], errors="coerce")

baseline_df = baseline_df.dropna(subset=["price", "title_group"])

gkf = GroupKFold(n_splits=5)

print("Development rows:", len(baseline_df))
print("Unique groups:", baseline_df["title_group"].nunique())
print("Number of folds:", gkf.get_n_splits())

Development rows: 5258
Unique groups: 2191
Number of folds: 5


In [28]:
baseline_df.shape

(5258, 40)

In [29]:
from pathlib import Path
import sys


In [31]:
PROJECT_PATH = Path.cwd().parent
sys.path.insert(0, str(PROJECT_PATH/"src"))


In [ ]:
import evaluation.baselines as baselines

importlib.reload(baselines)
from evaluation.baselines import (
    grouped_median_baseline
)

In [33]:
# E0 - Global Median
E0_predictions, E0_scores = grouped_median_baseline(data=baseline_df,group_col=None,prediction_col="pred_E0",n_splits=5)

# E1 - Brand Median
E1_predictions, E1_scores = grouped_median_baseline(data=baseline_df,group_col="brand",prediction_col="pred_E1",n_splits=5)

# E2 - Model Median
E2_predictions, E2_scores = grouped_median_baseline(data=baseline_df,group_col="model_key",prediction_col="pred_E2",n_splits=5)

# E3 - Family Median
E3_predictions, E3_scores = grouped_median_baseline(data=baseline_df,group_col="family",prediction_col="pred_E3",n_splits=5)


In [34]:

baseline_scores = pd.DataFrame([
    {"Experiment": "E0 - Global Median", **E0_scores},
    {"Experiment": "E1 - Brand Median", **E1_scores},
    {"Experiment": "E2 - Model Median", **E2_scores},
    {"Experiment": "E3 - Family Median", **E3_scores}
])

baseline_scores = baseline_scores.sort_values(by="mae").reset_index(drop=True)

display(baseline_scores)


,Experiment,mae,median_ae
0,E2 - Model Median,6090.148916,3000.0
1,E3 - Family Median,12592.910232,8075.0
2,E1 - Brand Median,13563.904526,9000.0
3,E0 - Global Median,15119.141308,11000.0


# Tf-IDF

to convert phone-ad titles into numerical features that a machine learning model can understand.

then use Ridge Regression to learn relationships between these numerical text features and phone prices.

In [35]:
import importlib
import evaluation.baselines as baselines
importlib.reload(baselines)

<module 'evaluation.baselines' from 'e:\\DEPI-Grad\\DEPI-GRADUATION-PROJECT\\src\\evaluation\\baselines.py'>

In [36]:
alpha_values = [0.1, 1, 10, 100, 1000]
alpha_results = []

for alpha in alpha_values:
    print(f"\nTesting alpha = {alpha}")

    _, scores = baselines.grouped_tfidf_ridge_baseline(
        data=baseline_df,
        alpha=alpha
    )

    alpha_results.append({
        "Alpha": alpha,
        "MAE": scores["tfidf_ridge_mae"],
        "MedianAE": scores["tfidf_ridge_median_ae"]
    })

alpha_results_df = pd.DataFrame(alpha_results)

alpha_results_df = alpha_results_df.sort_values(
    by="MAE"
).reset_index(drop=True)
best_alpha = alpha_results_df.iloc[0]["Alpha"]
print("Best Alpha:", best_alpha)



Testing alpha = 0.1
Fold 1: MAE = 8435.33
Fold 2: MAE = 8269.99
Fold 3: MAE = 7668.17
Fold 4: MAE = 7838.54
Fold 5: MAE = 7669.79

Testing alpha = 1
Fold 1: MAE = 7649.68
Fold 2: MAE = 6805.16
Fold 3: MAE = 7131.62
Fold 4: MAE = 7260.35
Fold 5: MAE = 6854.64

Testing alpha = 10
Fold 1: MAE = 9168.30
Fold 2: MAE = 7377.79
Fold 3: MAE = 8295.75
Fold 4: MAE = 7840.05
Fold 5: MAE = 7473.95

Testing alpha = 100
Fold 1: MAE = 12008.68
Fold 2: MAE = 10666.22
Fold 3: MAE = 11839.02
Fold 4: MAE = 11339.05
Fold 5: MAE = 10914.97

Testing alpha = 1000
Fold 1: MAE = 15000.50
Fold 2: MAE = 13781.28
Fold 3: MAE = 15729.33
Fold 4: MAE = 14908.85
Fold 5: MAE = 13633.24
Best Alpha: 1.0


In [37]:
pred_E4_df, tfidf_ridge_scores = (
    baselines.grouped_tfidf_ridge_baseline(
        data=baseline_df,
        alpha=best_alpha
    )
)

print(tfidf_ridge_scores)


Fold 1: MAE = 7649.68
Fold 2: MAE = 6805.16
Fold 3: MAE = 7131.62
Fold 4: MAE = 7260.35
Fold 5: MAE = 6854.64
{'tfidf_ridge_mae': 7140.322140557943, 'tfidf_ridge_median_ae': 4574.602142046555}


In [38]:
baseline_scores = pd.DataFrame([
    {"Experiment": "E0 - Global Median", **E0_scores},
    {"Experiment": "E1 - Brand Median", **E1_scores},
    {"Experiment": "E2 - Model Median", **E2_scores},
    {"Experiment": "E3 - Family Median", **E3_scores},
    {
        "Experiment": "E4 - TF-IDF + Ridge",
        "mae": tfidf_ridge_scores["tfidf_ridge_mae"],
        "median_ae": tfidf_ridge_scores["tfidf_ridge_median_ae"]
    }
])
baseline_scores = baseline_scores.sort_values(by="mae").reset_index(drop=True)
display(baseline_scores)


,Experiment,mae,median_ae
0,E2 - Model Median,6090.148916,3000.000000
1,E4 - TF-IDF + Ridge,7140.322141,4574.602142
2,E3 - Family Median,12592.910232,8075.000000
3,E1 - Brand Median,13563.904526,9000.000000
4,E0 - Global Median,15119.141308,11000.000000
